# Comparación final: ¿qué modelo con negativos queda?

**Para qué.** El profesor va a probar el sistema con un estudio que puede tener pelvis o no. Hay que cumplir las dos cosas a la vez: separar bien los fragmentos cuando hay pelvis y no inventar fragmentos cuando no la hay. Este notebook aplica la regla acordada antes de ver los resultados:

> **Candidatos:** modelo ajustado con negativos (`Negativos_ajuste`) y modelo entrenado desde cero con negativos (`Entrenamiento_negativos_completo`). El original es la referencia.
> Un candidato **es aceptable** si cumple las tres condiciones:
> 1. **Topes en 2D:** frente al original no baja más de 0,01 en mAP@0.50 ni en Dice, y conserva al menos el 99 % de los cortes con pelvis.
> 2. **Con pelvis (3D):** su F1 por fragmento, con la salida 3D reajustada para él, no queda más de un fragmento (0,04) por debajo del original.
> 3. **Sin pelvis:** el sistema completo (modelo + control de "pelvis presente") entrega 0 fragmentos en los tramos sin pelvis de los 5 casos de validación.
>
> Entre los aceptables gana el de mayor F1 3D; si la diferencia es de un fragmento o menos, gana el que invente menos fragmentos **sin** ayuda del control. Si ninguno es aceptable, se lleva la tabla al equipo.

**Cómo se mide, igual para los tres modelos:**
- Cada volumen de `val` pasa completo por el pipeline de `salida_3d` (todos los cortes, detección + NMS, decoder, geometría original, asociación 3D).
- La configuración 3D se **reajusta para cada modelo** con el mismo procedimiento de `salida_3d` (§10.5), así ninguno compite con una configuración hecha para otro.
- **Estudios sin pelvis:** los tramos de cada volumen antes y después de la pelvis (fémur, columna), recortados y pasados por el mismo pipeline como si fueran estudios separados.
- **Control de "pelvis presente"** (`src/control_pelvis.py`): hay pelvis si los cortes con una caja de score ≥ 0,2 forman un bloque continuo de al menos 80 mm. Si no, el sistema no entrega fragmentos. Si el control rechazara un caso que sí tiene pelvis, ese caso cuenta con 0 fragmentos predichos en el F1 del sistema.

Las funciones de `salida_3d.ipynb` se cargan por nombre (como hace `app/inferencia.py`); no se copian ni se modifica ese notebook. Este notebook no reemplaza ningún modelo.

**Cómo correrlo:** primero `Entrenamiento_negativos_completo` (deja el tercer modelo y las métricas 2D). Después, aquí: `SMOKE_TEST = True` y *Run All* (1 caso, ajuste corto), y luego `SMOKE_TEST = False`, reiniciar el kernel y *Run All* (unos 25-30 minutos).

## 1. Configuración y piezas del pipeline

In [ ]:
import gc, json, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "requirements.txt").is_file())
for _d in (ROOT / "src", ROOT):            # los cuadernos importan `rutas`, `deteccion_grid`... sin el prefijo src.
    if str(_d) not in sys.path:
        sys.path.insert(0, str(_d))
warnings.filterwarnings("ignore", message="'mode' parameter is deprecated")

from definiciones_notebooks import cargar_definiciones
from rutas import DATASET_CSV, METRICAS_DIR, MODELOS_DIR, NOTEBOOKS_DIR
from src import control_pelvis as cp
from src import cortes_vacios as cv

SMOKE_TEST = True          # 1) True: 1 caso y ajuste corto, para probar. 2) False: la comparación real.
BASE_DIR = ROOT            # lo usa resolve_repo_path

# Mismas definiciones y prioridad de cuadernos que salida_3d.ipynb y app/inferencia.py
DEFINICIONES_EQUIPO = {"resolve_repo_path", "apply_hu_window", "enhance_bone_contrast", "instance_to_semantic",
                       "boxes_from_semantic", "letterbox_pair", "FundidoraPC", "FundidoraPCPlus", "build_backbone",
                       "build_instance_segmentation_targets", "build_detection_region_priors",
                       "SegmentationConvBlock", "backbone_feature_pyramid", "SkipFusionBlock",
                       "CenterOffsetSkipDecoder", "restore_letterbox_label_mask"}
VALORES_EQUIPO = {"CLASS_ID_TO_NAME", "TARGET_COLORS"}
PRIORIDAD = ["Segmentacion_instancias", "Backbone", "Ventaneo_hu", "Targets_deteccion", "Cabeza_grid", "Dataset_splits",
             "nms_pertenencia_pipeline", "Entrenamiento_multitarea", "cbam"]
DEFINICIONES_SALIDA_3D = {"build_components_from_export", "trunk_features", "load_case_volumes", "preprocess_slices",
                          "empty_bundle", "infer_case", "_logit", "_sigmoid", "_pick_peaks", "boxes_allowed_mask",
                          "decode_cached_slice", "active_slices", "decode_and_restore", "link_instances_3d",
                          "clean_tracks", "class_of_gt_id", "contingency", "evaluate_volume_3d", "summarize_counts",
                          "sum_counts", "assemble_case", "CaseData", "evaluate_config", "_product", "tune_pipeline"}
VALORES_SALIDA_3D = {"DEVICE", "AMP", "IMAGE_SIZE", "CFG_SLICE", "CFG_DECODE", "CFG_LINK", "CFG_POST", "REGION_OF_CLASS",
                     "SIZE_BINS_VOX", "SIZE_BIN_NAMES", "MATCH_IOU", "DEFAULT_TUNE_PLAN"}

cuadernos = [NOTEBOOKS_DIR / f"{n}.ipynb" for n in PRIORIDAD if (NOTEBOOKS_DIR / f"{n}.ipynb").is_file()]
cargar_definiciones(cuadernos, DEFINICIONES_EQUIPO, VALORES_EQUIPO, destino=globals())
from cbam_modulos import CBAM, CompuertaResidualCBAM
cargar_definiciones(NOTEBOOKS_DIR / "salida_3d.ipynb", DEFINICIONES_SALIDA_3D, VALORES_SALIDA_3D, destino=globals())

df = pd.read_csv(DATASET_CSV, dtype={"case_id": str})       # lo usa load_case_volumes
CASOS_VAL = sorted(df.loc[df["split"] == "val", "case_id"])[: (1 if SMOKE_TEST else None)]
BASE_CFG_3D = {"decode": dict(CFG_DECODE), "link": dict(CFG_LINK), "post": dict(CFG_POST)}   # punto de partida de salida_3d §10.5
PLAN = DEFAULT_TUNE_PLAN[:1] if SMOKE_TEST else DEFAULT_TUNE_PLAN

_respaldo = MODELOS_DIR / "entrenamiento_multitarea" / "modelo_multitarea_final_sin_negativos.pt"
MODELOS = {
    "original": _respaldo if _respaldo.is_file() else MODELOS_DIR / "entrenamiento_multitarea" / "modelo_multitarea_final.pt",
    "ajustado": MODELOS_DIR / "entrenamiento_negativos" / "modelo_multitarea_negativos.pt",
    "completo": MODELOS_DIR / "entrenamiento_negativos_completo" / "modelo_multitarea_negativos_completo.pt",
}
for _n, _r in list(MODELOS.items()):
    if not _r.is_file() or _r.stat().st_size < 10_000:
        if SMOKE_TEST and _n == "completo":
            print(f"(SMOKE_TEST) Falta {_r.name}: se compara sin el modelo desde cero.")
            MODELOS.pop(_n)
            continue
        raise FileNotFoundError(f"Falta {_r} (o es un puntero de Git LFS: git lfs pull)")

SALIDA = METRICAS_DIR / ("comparacion_final_negativos_smoke" if SMOKE_TEST else "comparacion_final_negativos")
SALIDA.mkdir(parents=True, exist_ok=True)
print(f"Dispositivo: {DEVICE} | AMP: {AMP} | casos: {CASOS_VAL} | pasos de ajuste 3D: {len(PLAN)}")
print("Modelos:", {n: r.name for n, r in MODELOS.items()})
print(f"Control de pelvis: score >= {cp.DET_GATE} en un bloque continuo de >= {cp.MIN_MM:.0f} mm")

## 2. Evaluación de un modelo

Para cada modelo: inferencia en los volúmenes completos y en sus tramos sin pelvis, control de pelvis en todos, ajuste de la salida 3D con los volúmenes completos (mismo plan que `salida_3d` §10.5, a la mitad de resolución) y evaluación final a resolución completa. Los casos se sueltan de memoria al terminar cada modelo.

In [ ]:
def soltar_decodificado(case):
    # CaseData guarda la última decodificación (cientos de MB a resolución completa): se suelta después de evaluar
    case._decoded, case._decoded_key, case._gt_strided = None, None, {}
    gc.collect()


def evaluar_un_caso(case, cfg):
    _, conteos = evaluate_config([case], cfg, stride=1)
    soltar_decodificado(case)
    return conteos[0]


def evaluar_modelo(nombre, ruta):
    t0 = time.time()
    comp, _ = build_components_from_export(ruta)
    casos, controles, tramos = [], [], []
    for case_id in CASOS_VAL:
        volume, label, header = load_case_volumes(case_id, with_label=True, with_image=True)
        sp_z = float(header.GetSpacing()[2])
        bundle = infer_case(comp, case_id, CFG_SLICE, volume=volume, image_itk=header, verbose=False)
        controles.append({"modelo": nombre, "caso": case_id, "tramo": "completo", "cortes": volume.shape[0],
                          **cp.pelvis_presente_bundle(bundle, sp_z)})
        tiene = (label > 0).any(axis=(1, 2))
        for posicion, ini, fin in cv.tramos_sin_pelvis(tiene):
            sub = infer_case(comp, f"{case_id}_{posicion}", CFG_SLICE, volume=volume[ini:fin], image_itk=header, verbose=False)
            tramos.append({"modelo": nombre, "caso": case_id, "tramo": posicion, "cortes": fin - ini,
                           "mm": (fin - ini) * sp_z, "bundle": sub, "control": cp.pelvis_presente_bundle(sub, sp_z)})
        del volume
        casos.append(CaseData(bundle, label, header))
        del bundle, label
        gc.collect()
    print(f"[{nombre}] inferencia: {len(casos)} volúmenes y {len(tramos)} tramos sin pelvis ({time.time() - t0:.0f} s)")

    cfg_m, historial = tune_pipeline(casos, BASE_CFG_3D, plan=PLAN, stride=2, rounds=1, verbose=False)
    historial.to_csv(SALIDA / f"ajuste_3d_{nombre}.csv", index=False)
    (SALIDA / f"config_3d_{nombre}.json").write_text(json.dumps(cfg_m, indent=2, default=str), encoding="utf-8")
    for case in casos:
        soltar_decodificado(case)
    print(f"[{nombre}] ajuste 3D listo ({time.time() - t0:.0f} s)")

    filas_casos, conteos_modelo, conteos_sistema = [], [], []
    for case, ctrl in zip(casos, controles):
        k = evaluar_un_caso(case, cfg_m)
        k_sis = k if ctrl["presente"] else evaluate_volume_3d(np.zeros_like(case.gt), {}, case.gt)["counts"]
        conteos_modelo.append(k)
        conteos_sistema.append(k_sis)
        filas_casos.append({"modelo": nombre, "caso": case.bundle["case_id"], "control_presente": ctrl["presente"],
                            "racha_mm": round(ctrl["racha_mm"], 1), **summarize_counts(k_sis)})
    total = summarize_counts(sum_counts(conteos_sistema))
    total_modelo = summarize_counts(sum_counts(conteos_modelo))
    filas_casos.append({"modelo": nombre, "caso": "TODOS", "control_presente": all(c["presente"] for c in controles),
                        "racha_mm": np.nan, **total})

    filas_tramos = []
    for t in tramos:
        res = assemble_case(t.pop("bundle"), cfg_m["decode"], cfg_m["link"], cfg_m["post"], stride=1)
        n = int(len(res["table"]))
        filas_tramos.append({"modelo": nombre, "caso": t["caso"], "tramo": t["tramo"], "cortes": t["cortes"],
                             "mm": round(t["mm"], 1), "racha_mm": round(t["control"]["racha_mm"], 1),
                             "control_presente": t["control"]["presente"], "fragmentos_modelo": n,
                             "fragmentos_sistema": n if t["control"]["presente"] else 0})
        del res
    resumen = {"modelo": nombre, "f1_3d": total["f1@0.5"], "f1_3d_sin_control": total_modelo["f1@0.5"],
               "n_pred": total["n_pred"], "n_gt": total["n_gt"], "precision@0.5": total["precision@0.5"],
               "recall@0.5": total["recall@0.5"], "dice_regiones": total["dice_regiones"],
               "casos_con_pelvis_rechazados": int(sum(not c["presente"] for c in controles)),
               "frag_sin_pelvis_modelo": int(sum(f["fragmentos_modelo"] for f in filas_tramos)),
               "frag_sin_pelvis_sistema": int(sum(f["fragmentos_sistema"] for f in filas_tramos)),
               "tramos_sin_pelvis": len(filas_tramos), "minutos": (time.time() - t0) / 60}
    del comp, casos
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"[{nombre}] F1 3D {resumen['f1_3d']:.3f} | {resumen['n_pred']} fragmentos (reales {resumen['n_gt']}) | "
          f"sin pelvis: {resumen['frag_sin_pelvis_modelo']} fragmentos sin control, {resumen['frag_sin_pelvis_sistema']} con control "
          f"| {resumen['minutos']:.1f} min")
    return resumen, pd.DataFrame(filas_casos), pd.DataFrame(filas_tramos), pd.DataFrame(controles)

## 3. Los tres modelos, uno a la vez

In [ ]:
resumenes, por_caso, por_tramo, controles_completos = [], [], [], []
for nombre, ruta in MODELOS.items():
    r, casos_df, tramos_df, ctrl_df = evaluar_modelo(nombre, ruta)
    resumenes.append(r); por_caso.append(casos_df); por_tramo.append(tramos_df); controles_completos.append(ctrl_df)

resumen_3d = pd.DataFrame(resumenes).set_index("modelo")
tabla_casos = pd.concat(por_caso, ignore_index=True)
tabla_tramos = pd.concat(por_tramo, ignore_index=True)
resumen_3d.to_csv(SALIDA / "resumen_3d_val.csv")
tabla_casos.to_csv(SALIDA / "por_caso_val.csv", index=False)
tabla_tramos.to_csv(SALIDA / "tramos_sin_pelvis_val.csv", index=False)
pd.concat(controles_completos, ignore_index=True).to_csv(SALIDA / "control_pelvis_volumenes_val.csv", index=False)

print("Resumen en 3D (salida reajustada para cada modelo):")
display(resumen_3d.round(3))
print("Estudios sin pelvis: fragmentos que entrega cada modelo sin el control y con el control")
display(tabla_tramos.pivot_table(index=["caso", "tramo", "cortes"], columns="modelo",
                                 values=["fragmentos_modelo", "fragmentos_sistema"], aggfunc="sum"))
print("F1 3D por caso (sistema completo):")
display(tabla_casos.pivot_table(index="caso", columns="modelo", values="f1@0.5").round(3))

## 4. Aplicar la regla

Las métricas 2D (mAP, Dice y cortes con pelvis conservados) salen de `Entrenamiento_negativos_completo`, medidas con el mismo código para los tres modelos. El F1 3D y los fragmentos sin pelvis salen de la sección 3.

In [ ]:
D2 = METRICAS_DIR / "entrenamiento_negativos_completo"
_cmp2d = D2 / "comparacion_2d_val.csv"
_vol = D2 / "resumen_volumen_completo_val.csv"
if _cmp2d.is_file() and _vol.is_file():
    m2d = pd.read_csv(_cmp2d)
    m2d["modelo"] = m2d["modelo"].str.replace(" (referencia)", "", regex=False)
    m2d = m2d.set_index("modelo")
    vol = pd.read_csv(_vol, index_col=0)

    def metricas(nombre):
        return {"map50": float(m2d.loc[nombre, "val_det_map50"]), "dice": float(m2d.loc[nombre, "val_seg_dice_pred"]),
                "recall_pelvis": float(vol.loc[nombre, "recall_pelvis_cal"]), "f1_3d": float(resumen_3d.loc[nombre, "f1_3d"]),
                "frag_sin_pelvis_sistema": int(resumen_3d.loc[nombre, "frag_sin_pelvis_sistema"]),
                "frag_sin_pelvis_modelo": int(resumen_3d.loc[nombre, "frag_sin_pelvis_modelo"])}

    referencia = metricas("original")
    candidatos = {n: metricas(n) for n in MODELOS if n != "original"}
    decision = cv.regla_modelo_final(candidatos, referencia)
    tabla_regla = pd.DataFrame({n: {**candidatos[n], **decision["detalle"][n]} for n in candidatos}).T
    display(pd.concat([pd.DataFrame({"original": referencia}).T, tabla_regla]).round(3))
    (SALIDA / "decision_final.json").write_text(json.dumps({
        "smoke_test": SMOKE_TEST, "regla": cv.REGLA_FINAL, "control_pelvis": {"det_gate": cp.DET_GATE, "min_mm": cp.MIN_MM},
        "referencia": referencia, "candidatos": candidatos, **decision}, indent=2, default=float), encoding="utf-8")
    print("GANADOR:", decision["ganador"] or "ninguno (decide el equipo)", "|", decision["motivo"])
else:
    decision = None
    print("Faltan las métricas 2D de Entrenamiento_negativos_completo: correr ese notebook primero (en SMOKE_TEST se omite la regla).")

## 5. Qué sigue

1. Subir a la rama: este notebook ejecutado y `resultados/metricas/comparacion_final_negativos/` (no la carpeta `_smoke`).
2. Si hay ganador: adoptarlo (copiar su `.pt` sobre `resultados/modelos/entrenamiento_multitarea/modelo_multitarea_final.pt`, después de guardar el original como `modelo_multitarea_final_sin_negativos.pt`) y guardar su configuración 3D: `config_3d_<ganador>.json` de esta carpeta pasa a ser `resultados/metricas/salida_3d/config_3d.json` (o volver a correr `salida_3d` completo, que da la misma). Luego correr `nms_pertenencia_pipeline` (recalibra el umbral y reentrena la cabeza de pertenencia) y revisar el frontend.
3. Si no hay ganador: llevar las tablas de las secciones 3 y 4 al equipo.

**Para el informe.** La configuración 3D y la elección del modelo usan los mismos 5 casos de validación, así que estos números son optimistas; la medida definitiva es test, una sola vez. Los estudios sin pelvis son tramos vecinos de la pelvis (fémur, columna); el control de "pelvis presente" es la protección para estudios más lejanos (tórax, cabeza), que no están en el dataset.